Đọc dữ liệu và làm schaj

In [5]:
import pandas as pd
import numpy as np
import time
from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    KFold,
    ParameterSampler
)
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
df = pd.read_csv("../../data/raw/Diamonds Prices2022.csv")

# Bỏ cột không cần thiết
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Loại bỏ dòng có kích thước không hợp lệ
df = df[
    (df["x"] > 0) &
    (df["y"] > 0) &
    (df["z"] > 0)
]

# Xóa dòng trùng lặp
df = df.drop_duplicates().copy()

# Mã hóa thuộc tính phân loại
categorical_cols = ["cut", "color", "clarity"]

encoder = OrdinalEncoder()
df[categorical_cols] = encoder.fit_transform(
    df[categorical_cols]
)

# Tách đặc trưng và nhãn
X = df.drop(columns=["price"])
y = df["price"]

print("Số dòng sau tiền xử lý:", len(df))
print("Số thuộc tính đầu vào:", X.shape[1])
print("Số giá trị thiếu:", df.isnull().sum().sum())

Số dòng sau tiền xử lý: 53775
Số thuộc tính đầu vào: 9
Số giá trị thiếu: 0


Khởi tạo K-Fold và Randomized Search

In [7]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

param_grid = {
    "n_estimators": [100, 150, 200, 250],
    "max_depth": [10, 15, 20, 25, None],
    "min_samples_leaf": [1, 2, 3],
    "min_samples_split": [2, 5, 10],
    "max_features": [1.0, 0.8, 0.6]
}

# Lấy ngẫu nhiên 20 cấu hình, cố định để tái lập
param_list = list(ParameterSampler(
    param_grid,
    n_iter=20,
    random_state=42
))

print("Số Fold:", kf.get_n_splits())
print("Số cấu hình mỗi Fold:", len(param_list))
print("Tổng lượt tìm kiếm:", 5 * len(param_list))

Số Fold: 5
Số cấu hình mỗi Fold: 20
Tổng lượt tìm kiếm: 100


Huấn luyện và đánh giá 5 Fold

In [8]:
# Thư mục lưu kết quả
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

results = []
best_params_per_fold = []

start_time = time.time()

for fold, (train_idx, test_idx) in enumerate(
    kf.split(X), start=1
):

    print(f"\n========== FOLD {fold}/5 ==========", flush=True)

    # Chia Train và Test theo Fold
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    # Tách Validation từ Train
    X_subtrain, X_val, y_subtrain, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        random_state=42
    )

    best_rmse = float("inf")
    best_params = None

    # Tìm tham số tốt nhất
    for i, params in enumerate(param_list, start=1):

        model = RandomForestRegressor(
            **params,
            criterion="squared_error",
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_subtrain, y_subtrain)

        y_val_pred = model.predict(X_val)

        rmse_val = np.sqrt(
            mean_squared_error(y_val, y_val_pred)
        )

        if rmse_val < best_rmse:
            best_rmse = rmse_val
            best_params = params.copy()

        print(
            f"Fold {fold}: Đã thử {i}/20 cấu hình",
            flush=True
        )

    # Huấn luyện lại trên toàn bộ Train
    final_model = RandomForestRegressor(
        **best_params,
        criterion="squared_error",
        random_state=42,
        n_jobs=-1
    )

    final_model.fit(X_train, y_train)

    # Dự đoán
    y_train_pred = final_model.predict(X_train)
    y_test_pred = final_model.predict(X_test)

    # Đánh giá
    mae_train = mean_absolute_error(y_train, y_train_pred)
    mse_train = mean_squared_error(y_train, y_train_pred)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(y_train, y_train_pred)

    mae_test = mean_absolute_error(y_test, y_test_pred)
    mse_test = mean_squared_error(y_test, y_test_pred)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_test, y_test_pred)

    results.append({
        "Fold": fold,
        "MAE Train": mae_train,
        "MSE Train": mse_train,
        "RMSE Train": rmse_train,
        "R2 Train": r2_train,
        "MAE Test": mae_test,
        "MSE Test": mse_test,
        "RMSE Test": rmse_test,
        "R2 Test": r2_test
    })

    best_params_per_fold.append({
        "Fold": fold,
        **best_params,
        "RMSE Validation": best_rmse
    })

    # Lưu tiến độ sau mỗi Fold
    pd.DataFrame(results).to_csv(
        output_dir / "RF_Level5_KFold_Progress.csv",
        index=False
    )

    pd.DataFrame(best_params_per_fold).to_csv(
        output_dir / "RF_Level5_KFold_Params.csv",
        index=False
    )

    print("\nTham số tốt nhất:", best_params)
    print(f"RMSE Validation: {best_rmse:.4f}")
    print(f"RMSE Test Fold {fold}: {rmse_test:.4f}")
    print(f"Hoàn thành Fold {fold}/5", flush=True)

train_time = time.time() - start_time

print(f"\nTổng thời gian: {train_time:.4f} giây")


========== FOLD 1/5 ==========
Fold 1: Đã thử 1/20 cấu hình
Fold 1: Đã thử 2/20 cấu hình
Fold 1: Đã thử 3/20 cấu hình
Fold 1: Đã thử 4/20 cấu hình
Fold 1: Đã thử 5/20 cấu hình
Fold 1: Đã thử 6/20 cấu hình
Fold 1: Đã thử 7/20 cấu hình
Fold 1: Đã thử 8/20 cấu hình
Fold 1: Đã thử 9/20 cấu hình
Fold 1: Đã thử 10/20 cấu hình
Fold 1: Đã thử 11/20 cấu hình
Fold 1: Đã thử 12/20 cấu hình
Fold 1: Đã thử 13/20 cấu hình
Fold 1: Đã thử 14/20 cấu hình
Fold 1: Đã thử 15/20 cấu hình
Fold 1: Đã thử 16/20 cấu hình
Fold 1: Đã thử 17/20 cấu hình
Fold 1: Đã thử 18/20 cấu hình
Fold 1: Đã thử 19/20 cấu hình
Fold 1: Đã thử 20/20 cấu hình

Tham số tốt nhất: {'n_estimators': 150, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': None}
RMSE Validation: 560.6880
RMSE Test Fold 1: 529.8388
Hoàn thành Fold 1/5

========== FOLD 2/5 ==========
Fold 2: Đã thử 1/20 cấu hình
Fold 2: Đã thử 2/20 cấu hình
Fold 2: Đã thử 3/20 cấu hình
Fold 2: Đã thử 4/20 cấu hình
Fold 2: Đã thử 5/20 cấu hình

Lưu KQ

In [10]:
final_results = pd.DataFrame(results)

average_row = {"Fold": "Average"}

for col in final_results.columns:
    if col != "Fold":
        average_row[col] = final_results[col].mean()

summary = pd.concat(
    [final_results, pd.DataFrame([average_row])],
    ignore_index=True
)

best_params_df = pd.DataFrame(best_params_per_fold)

print("KẾT QUẢ RANDOM FOREST K-FOLD MỨC 5")
print(
    summary[
        ["Fold", "RMSE Train", "RMSE Test", "R2 Test"]
    ].to_string(index=False)
)

print("\nTHAM SỐ TỐT NHẤT TỪNG FOLD")
print(best_params_df.to_string(index=False))

print(f"\nTổng thời gian: {train_time:.4f} giây")

KẾT QUẢ RANDOM FOREST K-FOLD MỨC 5
   Fold  RMSE Train  RMSE Test  R2 Test
      1  203.534536 529.838777 0.982236
      2  201.861732 538.822912 0.981345
      3  201.016484 536.247952 0.981864
      4  204.333890 545.760145 0.981669
      5  200.991241 574.679832 0.979306
Average  202.347577 545.069924 0.981284

THAM SỐ TỐT NHẤT TỪNG FOLD
 Fold  n_estimators  min_samples_split  min_samples_leaf  max_features max_depth  RMSE Validation
    1           150                  2                 1           0.8      None       560.688020
    2           150                  2                 1           0.8      None       535.140889
    3           150                  2                 1           0.8      None       529.632264
    4           150                  2                 1           0.8      None       543.365139
    5           150                  2                 1           0.8      None       537.791466

Tổng thời gian: 466.9626 giây
